In [9]:
import scipy.io as sio
import numpy as np
import pandas as pd
import glob
import os
import re
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert
from scipy.fft import fft, fftfreq

def load_mat_file(filepath, default_rpm=1772):
    data = sio.loadmat(filepath)
    de_key = [k for k in data if 'DE_time' in k][0]
    rpm_keys = [k for k in data if 'RPM' in k]
    signal = data[de_key].flatten()
    if rpm_keys:
        rpm = int(data[rpm_keys[0]][0, 0])
    else:
        rpm = default_rpm  # Normal file doesn't report RPM; assumed same as other load-1 files
    return signal, rpm

def parse_label(filename):
    name = os.path.basename(filename).replace(".mat", "")
    if "Normal" in name:
        return "Normal", None
    match = re.match(r"([A-Za-z]+)(\d+)", name)
    if match:
        return match.group(1), match.group(2)
    return "Unknown", None

In [10]:
data_dir = "../data/raw/cwru"
mat_files = glob.glob(os.path.join(data_dir, "*.mat"))

summary = []
for f in mat_files:
    signal, rpm = load_mat_file(f)
    fault_type, severity = parse_label(f)
    summary.append({
        "file": os.path.basename(f),
        "fault_type": fault_type,
        "severity": severity,
        "rpm": rpm,
        "n_samples": len(signal),
        "duration_sec": len(signal) / 48000
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

                    file fault_type severity   rpm  n_samples  duration_sec
0         B007_1_123.mat          B      007  1772     487384     10.153833
1         B014_1_190.mat          B      014  1772     486224     10.129667
2         B021_1_227.mat          B      021  1774     486804     10.141750
3        IR007_1_110.mat         IR      007  1772     486224     10.129667
4        IR014_1_175.mat         IR      014  1772     489125     10.190104
5        IR021_1_214.mat         IR      021  1774     485063     10.105479
6      OR007_6_1_136.mat         OR      007  1774     486804     10.141750
7      OR014_6_1_202.mat         OR      014  1772     484483     10.093396
8      OR021_6_1_239.mat         OR      021  1771     489125     10.190104
9  Time_Normal_1_098.mat     Normal      NaN  1772     483903     10.081312


In [11]:
def compute_bearing_frequencies(rpm, n_balls=9, d=7.938, D=38.5, theta=0):
    fr = rpm / 60
    ratio = (d / D) * np.cos(np.radians(theta))
    BPFO = (n_balls / 2) * fr * (1 - ratio)
    BPFI = (n_balls / 2) * fr * (1 + ratio)
    BSF  = (D / (2 * d)) * fr * (1 - ratio**2)
    FTF  = (fr / 2) * (1 - ratio)
    return {"BPFO": BPFO, "BPFI": BPFI, "BSF": BSF, "FTF": FTF, "fr": fr}

def envelope_spectrum(signal, fs=48000, band=(2500, 3500)):
    nyq = fs / 2
    b, a = butter(4, [band[0]/nyq, band[1]/nyq], btype='band')
    filtered = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered))
    envelope = envelope - envelope.mean()
    n = len(envelope)
    yf = np.abs(fft(envelope))[:n // 2] / n
    xf = fftfreq(n, 1/fs)[:n // 2]
    return xf, yf

def energy_near_frequency(xf, yf, target_freq, window=3):
    mask = (xf > target_freq - window) & (xf < target_freq + window)
    if not mask.any():
        return 0.0
    return yf[mask].max()

In [12]:
signal, rpm = load_mat_file("../data/raw/cwru/B007_1_123.mat")
freqs = compute_bearing_frequencies(rpm)
xf, yf = envelope_spectrum(signal)

for name in ["BPFO", "BPFI", "BSF"]:
    for harmonic in [1, 2, 3]:
        target = freqs[name] * harmonic
        amp = energy_near_frequency(xf, yf, target)
        print(f"{name} x{harmonic} ({target:.1f} Hz): {amp:.6f}")

BPFO x1 (105.5 Hz): 0.003396
BPFO x2 (211.0 Hz): 0.004748
BPFO x3 (316.5 Hz): 0.003035
BPFI x1 (160.3 Hz): 0.002349
BPFI x2 (320.6 Hz): 0.003035
BPFI x3 (480.9 Hz): 0.002098
BSF x1 (68.6 Hz): 0.003038
BSF x2 (137.2 Hz): 0.002465
BSF x3 (205.7 Hz): 0.001837


In [7]:
data_normal = sio.loadmat("../data/raw/cwru/Time_Normal_1_098.mat")
print([k for k in data_normal.keys() if not k.startswith("__")])

['X098_DE_time', 'X098_FE_time']


In [13]:
def extract_window_features(window, rpm, fs=48000):
    """One window of raw signal -> one row of frequency-domain features."""
    freqs = compute_bearing_frequencies(rpm)
    xf, yf = envelope_spectrum(window, fs=fs)

    features = {}
    for name in ["BPFO", "BPFI", "BSF"]:
        for harmonic in [1, 2, 3]:
            target = freqs[name] * harmonic
            features[f"{name}_x{harmonic}"] = energy_near_frequency(xf, yf, target)

    # A couple of simple time-domain stats too, cheap to add, often genuinely useful alongside frequency features
    features["rms"] = np.sqrt(np.mean(window**2))
    features["kurtosis"] = pd.Series(window).kurtosis()

    return features

def process_file(filepath, window_size=2048):
    signal, rpm = load_mat_file(filepath)
    fault_type, severity = parse_label(filepath)

    n_windows = len(signal) // window_size
    rows = []
    for i in range(n_windows):
        window = signal[i*window_size : (i+1)*window_size]
        feats = extract_window_features(window, rpm)
        feats["fault_type"] = fault_type
        feats["severity"] = severity
        feats["source_file"] = os.path.basename(filepath)
        rows.append(feats)
    return rows

all_rows = []
for f in mat_files:
    all_rows.extend(process_file(f))

bearing_df = pd.DataFrame(all_rows)
print(bearing_df.shape)
print(bearing_df["fault_type"].value_counts())
print(bearing_df.head())

(2369, 14)
fault_type
B         711
IR        711
OR        711
Normal    236
Name: count, dtype: int64
   BPFO_x1   BPFO_x2  BPFO_x3  BPFI_x1  BPFI_x2  BPFI_x3    BSF_x1  BSF_x2  \
0      0.0  0.003697      0.0      0.0      0.0      0.0  0.014284     0.0   
1      0.0  0.011968      0.0      0.0      0.0      0.0  0.008986     0.0   
2      0.0  0.016038      0.0      0.0      0.0      0.0  0.002648     0.0   
3      0.0  0.017136      0.0      0.0      0.0      0.0  0.024676     0.0   
4      0.0  0.008595      0.0      0.0      0.0      0.0  0.008025     0.0   

   BSF_x3       rms  kurtosis fault_type severity     source_file  
0     0.0  0.124006 -0.036491          B      007  B007_1_123.mat  
1     0.0  0.134312 -0.075956          B      007  B007_1_123.mat  
2     0.0  0.151008 -0.269128          B      007  B007_1_123.mat  
3     0.0  0.158422  0.141028          B      007  B007_1_123.mat  
4     0.0  0.139922  0.410030          B      007  B007_1_123.mat  


In [14]:
def energy_near_frequency(xf, yf, target_freq, window=10):  # comfortably wider than 5.86Hz bin spacing
    mask = (xf > target_freq - window) & (xf < target_freq + window)
    if not mask.any():
        return 0.0
    return yf[mask].max()

def process_file(filepath, window_size=8192):
    signal, rpm = load_mat_file(filepath)
    fault_type, severity = parse_label(filepath)

    n_windows = len(signal) // window_size
    rows = []
    for i in range(n_windows):
        window = signal[i*window_size : (i+1)*window_size]
        feats = extract_window_features(window, rpm)
        feats["fault_type"] = fault_type
        feats["severity"] = severity
        feats["source_file"] = os.path.basename(filepath)
        rows.append(feats)
    return rows

all_rows = []
for f in mat_files:
    all_rows.extend(process_file(f))

bearing_df = pd.DataFrame(all_rows)
print(bearing_df.shape)
print(bearing_df["fault_type"].value_counts())
print(bearing_df.head())

(590, 14)
fault_type
B         177
IR        177
OR        177
Normal     59
Name: count, dtype: int64
    BPFO_x1   BPFO_x2   BPFO_x3   BPFI_x1   BPFI_x2   BPFI_x3    BSF_x1  \
0  0.008997  0.004664  0.006352  0.012708  0.006352  0.005670  0.012545   
1  0.008904  0.005346  0.009276  0.010316  0.009276  0.005901  0.011742   
2  0.007442  0.010293  0.005870  0.007488  0.005870  0.004328  0.012935   
3  0.008119  0.007268  0.005471  0.004528  0.005471  0.006511  0.012007   
4  0.013946  0.008752  0.007850  0.009225  0.008992  0.002989  0.008875   

     BSF_x2    BSF_x3       rms  kurtosis fault_type severity     source_file  
0  0.008826  0.003323  0.142582  0.059325          B      007  B007_1_123.mat  
1  0.012436  0.003518  0.143050  0.056565          B      007  B007_1_123.mat  
2  0.011052  0.008257  0.137513 -0.148703          B      007  B007_1_123.mat  
3  0.009532  0.007150  0.138330  0.413644          B      007  B007_1_123.mat  
4  0.005527  0.008752  0.144327  0.079928     

In [15]:
freq_cols = [c for c in bearing_df.columns if any(x in c for x in ["BPFO", "BPFI", "BSF"])]
zero_fraction = (bearing_df[freq_cols] == 0.0).mean()
print(zero_fraction)

BPFO_x1    0.0
BPFO_x2    0.0
BPFO_x3    0.0
BPFI_x1    0.0
BPFI_x2    0.0
BPFI_x3    0.0
BSF_x1     0.0
BSF_x2     0.0
BSF_x3     0.0
dtype: float64


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

feature_cols_bearing = [c for c in bearing_df.columns if c not in ["fault_type", "severity", "source_file"]]

X = bearing_df[feature_cols_bearing]
y = bearing_df["fault_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify keeps class balance consistent in both sets
)

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))
print()
print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(confusion_matrix(y_test, preds), index=clf.classes_, columns=clf.classes_))

Accuracy: 0.940677966101695

              precision    recall  f1-score   support

           B       0.85      0.97      0.91        35
          IR       0.97      1.00      0.99        35
      Normal       1.00      1.00      1.00        12
          OR       1.00      0.83      0.91        36

    accuracy                           0.94       118
   macro avg       0.96      0.95      0.95       118
weighted avg       0.95      0.94      0.94       118


Confusion matrix (rows=true, cols=predicted):
         B  IR  Normal  OR
B       34   1       0   0
IR       0  35       0   0
Normal   0   0      12   0
OR       6   0       0  30


In [17]:
importances_bearing = pd.DataFrame({
    "feature": feature_cols_bearing,
    "importance": clf.feature_importances_
}).sort_values("importance", ascending=False)

print(importances_bearing)

     feature  importance
9        rms    0.212539
8     BSF_x3    0.141218
1    BPFO_x2    0.132710
0    BPFO_x1    0.089183
7     BSF_x2    0.078171
3    BPFI_x1    0.072222
2    BPFO_x3    0.068827
6     BSF_x1    0.064384
4    BPFI_x2    0.062975
10  kurtosis    0.041438
5    BPFI_x3    0.036334


In [18]:
# Get the actual misclassified rows
test_results = X_test.copy()
test_results["true_label"] = y_test.values
test_results["predicted"] = preds

confused = test_results[(test_results["true_label"] == "OR") & (test_results["predicted"] == "B")]
print(confused[["BPFO_x1", "BPFO_x2", "BSF_x1", "BSF_x2", "rms", "kurtosis"]])

# Compare against typical values for each true class
print("\nTypical OR values:")
print(test_results[test_results["true_label"]=="OR"][["BPFO_x1", "BSF_x1"]].describe())
print("\nTypical B values:")
print(test_results[test_results["true_label"]=="B"][["BPFO_x1", "BSF_x1"]].describe())

      BPFO_x1   BPFO_x2    BSF_x1    BSF_x2       rms  kurtosis
429  0.006545  0.006165  0.008135  0.004535  0.130471 -0.177166
459  0.003145  0.006477  0.009701  0.008736  0.136963 -0.067922
445  0.011136  0.008065  0.011822  0.006556  0.140148  0.091155
443  0.008887  0.008799  0.012479  0.008838  0.140750 -0.006943
415  0.009128  0.006541  0.017189  0.005719  0.144158  0.278830
446  0.009458  0.006044  0.011024  0.006296  0.139986 -0.037791

Typical OR values:
         BPFO_x1     BSF_x1
count  36.000000  36.000000
mean    0.178671   0.077924
std     0.154247   0.057257
min     0.003145   0.006801
25%     0.009571   0.011648
50%     0.189222   0.077378
75%     0.251534   0.136879
max     0.423024   0.162054

Typical B values:
         BPFO_x1     BSF_x1
count  35.000000  35.000000
mean    0.014519   0.014936
std     0.010844   0.008568
min     0.004545   0.005131
25%     0.008828   0.010888
50%     0.013041   0.013496
75%     0.016073   0.015597
max     0.058162   0.051995


In [19]:
# Add window index and block grouping BEFORE splitting
def process_file(filepath, window_size=8192):
    signal, rpm = load_mat_file(filepath)
    fault_type, severity = parse_label(filepath)
    n_windows = len(signal) // window_size
    rows = []
    for i in range(n_windows):
        window = signal[i*window_size : (i+1)*window_size]
        feats = extract_window_features(window, rpm)
        feats["fault_type"] = fault_type
        feats["severity"] = severity
        feats["source_file"] = os.path.basename(filepath)
        feats["window_idx"] = i
        rows.append(feats)
    return rows

all_rows = []
for f in mat_files:
    all_rows.extend(process_file(f))

bearing_df = pd.DataFrame(all_rows)

# Group consecutive windows into blocks of 5 for voting
block_size = 5
bearing_df["block_id"] = bearing_df["source_file"] + "_" + (bearing_df["window_idx"] // block_size).astype(str)

# Split by BLOCK, not window - keeps voting groups intact on one side
unique_blocks = bearing_df["block_id"].unique()
train_blocks, test_blocks = train_test_split(unique_blocks, test_size=0.2, random_state=42)

train_df_bearing = bearing_df[bearing_df["block_id"].isin(train_blocks)]
test_df_bearing = bearing_df[bearing_df["block_id"].isin(test_blocks)]

print(f"Train windows: {len(train_df_bearing)}, Test windows: {len(test_df_bearing)}")
print(f"Train blocks: {len(train_blocks)}, Test blocks: {len(test_blocks)}")

Train windows: 473, Test windows: 117
Train blocks: 96, Test blocks: 24


In [20]:
X_train_b = train_df_bearing[feature_cols_bearing]
y_train_b = train_df_bearing["fault_type"]
X_test_b = test_df_bearing[feature_cols_bearing]
y_test_b = test_df_bearing["fault_type"]

clf2 = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf2.fit(X_train_b, y_train_b)

# Per-window predictions, same as before
window_preds = clf2.predict(X_test_b)
window_accuracy = accuracy_score(y_test_b, window_preds)
print("Per-window accuracy (block-based split):", window_accuracy)

# Now the majority-vote step: group predictions by block, take the most common prediction per block
test_df_bearing = test_df_bearing.copy()
test_df_bearing["prediction"] = window_preds

block_votes = test_df_bearing.groupby("block_id").agg(
    true_label=("fault_type", "first"),          # all windows in a block share the same true label
    predicted=("prediction", lambda x: x.mode()[0])  # majority vote across the block's windows
)

block_accuracy = accuracy_score(block_votes["true_label"], block_votes["predicted"])
print("Per-block (majority vote) accuracy:", block_accuracy)

print("\nBlock-level confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(block_votes["true_label"], block_votes["predicted"], labels=clf2.classes_),
    index=clf2.classes_, columns=clf2.classes_
))

Per-window accuracy (block-based split): 0.9401709401709402
Per-block (majority vote) accuracy: 0.9583333333333334

Block-level confusion matrix:
        B  IR  Normal  OR
B       7   0       0   0
IR      0  10       0   0
Normal  0   0       1   0
OR      1   0       0   5


In [ ]:
from sklearn.metrics import accuracy_score

lofo_results = []
unique_files = bearing_df["source_file"].unique()

for held_out_file in unique_files:
    train_lofo = bearing_df[bearing_df["source_file"] != held_out_file]
    test_lofo = bearing_df[bearing_df["source_file"] == held_out_file]

    X_tr, y_tr = train_lofo[feature_cols_bearing], train_lofo["fault_type"]
    X_te, y_te = test_lofo[feature_cols_bearing], test_lofo["fault_type"]

    model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)

    lofo_results.append({
        "held_out_file": held_out_file,
        "true_fault_type": test_lofo["fault_type"].iloc[0],
        "n_test_windows": len(test_lofo),
        "accuracy": acc
    })

lofo_df = pd.DataFrame(lofo_results)
print(lofo_df)
print(f"\nMean LOFO accuracy: {lofo_df['accuracy'].mean():.3f}")
print(f"Min LOFO accuracy: {lofo_df['accuracy'].min():.3f} (worst case — this is the honest number)")

In [21]:
from sklearn.metrics import accuracy_score

lofo_results = []
unique_files = bearing_df["source_file"].unique()

for held_out_file in unique_files:
    train_lofo = bearing_df[bearing_df["source_file"] != held_out_file]
    test_lofo = bearing_df[bearing_df["source_file"] == held_out_file]

    X_tr, y_tr = train_lofo[feature_cols_bearing], train_lofo["fault_type"]
    X_te, y_te = test_lofo[feature_cols_bearing], test_lofo["fault_type"]

    model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)

    lofo_results.append({
        "held_out_file": held_out_file,
        "true_fault_type": test_lofo["fault_type"].iloc[0],
        "n_test_windows": len(test_lofo),
        "accuracy": acc
    })

lofo_df = pd.DataFrame(lofo_results)
print(lofo_df)
print(f"\nMean LOFO accuracy: {lofo_df['accuracy'].mean():.3f}")
print(f"Min LOFO accuracy: {lofo_df['accuracy'].min():.3f} (worst case — this is the honest number)")

           held_out_file true_fault_type  n_test_windows  accuracy
0         B007_1_123.mat               B              59  0.152542
1         B014_1_190.mat               B              59  0.491525
2         B021_1_227.mat               B              59  0.949153
3        IR007_1_110.mat              IR              59  0.050847
4        IR014_1_175.mat              IR              59  0.000000
5        IR021_1_214.mat              IR              59  0.000000
6      OR007_6_1_136.mat              OR              59  1.000000
7      OR014_6_1_202.mat              OR              59  0.000000
8      OR021_6_1_239.mat              OR              59  0.000000
9  Time_Normal_1_098.mat          Normal              59  0.000000

Mean LOFO accuracy: 0.264
Min LOFO accuracy: 0.000 (worst case — this is the honest number)
